# 09 — Fatoração de matrizes e SVD

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 09/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/09_svd_fatoracao.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_09_svd_fatoracao.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Compreender fatores latentes
- Aplicar TruncatedSVD como demonstração, não como Surprise SVD

## Fundamentação conceitual
Na fatoração, aproximamos $R\approx PQ^T$. `TruncatedSVD` de uma matriz com ausências preenchidas por zero é **demonstração algébrica**, não equivalente ao SVD enviesado do pacote Surprise, que minimiza erro apenas nas avaliações observadas.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Fatores latentes em matriz pequena

In [2]:
from sklearn.decomposition import TruncatedSVD
popular=ratings.movieId.value_counts().head(250).index
M=ratings[ratings.movieId.isin(popular)].pivot_table(index='userId',columns='movieId',values='rating').fillna(0)
model=TruncatedSVD(n_components=12,random_state=42)
U=model.fit_transform(M); reconstruction=U @ model.components_
print('Original:',M.shape,'Fatores:',U.shape,'Reconstrução:',reconstruction.shape)

Original: (602, 250) Fatores: (602, 12) Reconstrução: (602, 250)


**Interpretação.** A matriz reconstruída não é automaticamente uma previsão calibrada de notas.

## Recomendações latentes demonstrativas

In [3]:
user=1; i=M.index.get_loc(user); scores=pd.Series(reconstruction[i],index=M.columns)
seen=set(ratings.loc[ratings.userId.eq(user),'movieId']); top=scores.drop(labels=list(seen.intersection(scores.index))).nlargest(5)
print(movies[movies.movieId.isin(top.index)][['movieId','title']].to_string(index=False))

 movieId                                     title
      32 Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
     589         Terminator 2: Judgment Day (1991)
    1036                           Die Hard (1988)
    1200                             Aliens (1986)
    2762                   Sixth Sense, The (1999)


**Interpretação.** Pontuações latentes geram candidatos, mas requerem teste offline honesto.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** fatores e regularização. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **fatores e regularização** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **fatores e regularização** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — SVD algébrica versus fatoração colaborativa

`TruncatedSVD` aplicada à matriz preenchida com zero minimiza uma aproximação matricial, **não** o erro apenas nas notas observadas. A fatoração colaborativa convencional ajusta fatores `p_u`, `q_i` e eventualmente vieses minimizando perda nos pares de treino observados. Portanto, não se deve tratar a reconstrução atual como uma previsão de notas validada.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [4]:
# Medir a reconstrução SOMENTE nos pares que já eram observados — diagnóstico de treino
observado = M.to_numpy() > 0
r = M.to_numpy()
rmse_observado_treino = np.sqrt(np.mean((reconstruction[observado]-r[observado])**2))
print('RMSE nos pares observados usados no próprio ajuste (NÃO teste):', round(float(rmse_observado_treino),3))
print('Esse valor não comprova generalização nem equivalência ao Surprise SVD.')

RMSE nos pares observados usados no próprio ajuste (NÃO teste): 1.937
Esse valor não comprova generalização nem equivalência ao Surprise SVD.


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Não interprete zeros imputados como avaliações reais nem reporte RMSE desse exemplo como qualidade geral.

## Exercícios
Compare os resultados para 6 e 20 componentes.

## Soluções comentadas
Altere `n_components` respeitando `min(M.shape)`; mais fatores podem aumentar sobreajuste.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [5]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
top.rename("score_latente_demonstrativo").rename_axis("movieId").reset_index().merge(
    movies[["movieId", "title"]], on="movieId", how="left").to_csv(
    out / "09_top_latente_demonstrativo.csv", index=False)
print('Outputs do capítulo 09 salvos em:', out.resolve())


Outputs do capítulo 09 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
